# Structured output
- Models can be requested to provide the response in a format matching a given schema, This is useful for ensuring the output can be easily parsed and used in subsequent procssing. Langchain supports multiple schema types and methods for enforcing structured output.

# Pydantic
 - Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7b11ef3202c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7b11eeebf7d0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [4]:
model_with_structured = model.with_structured_output(Movie)
model_with_structured

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7b11ef3202c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7b11eeebf7d0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the

In [5]:
model_with_structured.invoke("Provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [7]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

model_with_structured = model.with_structured_output(Movie,include_raw=True)   
response = model_with_structured.invoke("Provide details about the movie Inception")
response 

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants details about the movie Inception. The developer message indicates we have a tool "functions.Movie" that takes director, rating, title, year. But the user didn\'t specify what details to provide. We should likely call the function with the appropriate arguments for Inception: director Christopher Nolan, rating maybe 8.8 (from IMDb?), title "Inception", year 2010. But the function signature expects director, rating, title, year. We can call the function. The user didn\'t request a specific rating, but we can provide rating out of 10. Provide rating 8.8. So we call functions.Movie with director="Christopher Nolan", rating=8.8, title="Inception", year=2010. That will return something. We\'ll then respond with the details.', 'tool_calls': [{'id': 'fc_c0607879-2384-47ef-9f9c-36359c1d9286', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Mo

# Nested structure

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structured = model.with_structured_output(MovieDetails)
response = model_with_structured.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Satoshi Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160000000.0)

# TypedDict
- TypedDict provide a similar alternative using python's built in typing, ideal when you don't need runtime validation

In [11]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The movie's rating out of 10"]


model_with_typed_dict = model.with_structured_output(MovieDict)

response = model_with_typed_dict.invoke(
    "Please provide the details of the movie Avengers."
)

print(response)

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2008}


In [12]:


class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structured = model.with_structured_output(MovieDetails)
response = model_with_structured.invoke("Provide details about the movie Inception")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'}],
 'genres': ['Action', 'Science Fiction', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [13]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

In [14]:
import os
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [16]:
## DAta class

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """contact information for a person"""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo
)   
result = agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"Extract contact info from: Dhinesh kumar, dk@gmail.org, (555) 123-4567", 
        }
    ]
}) 

result["structured_response"]

ContactInfo(name='Dhinesh kumar', email='dk@gmail.org', phone='(555) 123-4567')